## 1. Check A100 accelerator

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), "CUDA not available. Enable an A100 GPU runtime in Colab."
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu, "VRAM GiB:", round(vram, 1))
subprocess.run(["nvidia-smi"], check=True)
if "A100" not in gpu:
    raise RuntimeError(f"Expected an A100, got {gpu}. Change the runtime or deliberately update this guard.")

GPU: NVIDIA A100-SXM4-40GB VRAM GiB: 39.5


## 2. Mount Drive and clone the exact source

In [ ]:
# headless run on probook: no Drive mount, outputs go to a local folder that colab-job fetches
from pathlib import Path
import os, json, shutil, subprocess, time, hashlib

REPO_URL = "https://github.com/utsab818/dpr-mining-study.git"
REPO = Path('/content/dpr-mining-study')
DRIVE_ROOT = Path('/content/dpr_v3_a100_seed42')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Git commit:', commit)
prior_commit_file = DRIVE_ROOT / 'source_commit.txt'
if prior_commit_file.exists():
    prior_commit = prior_commit_file.read_text().strip()
    if prior_commit != commit:
        raise RuntimeError(
            f'Drive root belongs to commit {prior_commit}, but cloned {commit}. '
            'Use a new DRIVE_ROOT for the new revision.'
        )
for name in ['data.py','model.py','loss.py','mine_negatives.py','train.py','evaluate.py','warmup_v3.py']:
    assert (REPO / 'src' / name).exists(), f'Missing src/{name} in GitHub. Push latest V3 code first.'
print('All expected source files found.')

Git commit: fde8abd4a6ee39965877f9f61547215b7c6b1727
All expected source files found.


## 3. Install dependencies and record environment

In [ ]:
# Keep the Colab-provided CUDA-compatible torch build.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.35,<5', 'datasets>=2.16,<5', 'safetensors'], check=True)
import transformers, datasets
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('cuda:', torch.version.cuda)
(DRIVE_ROOT / 'environment.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
(DRIVE_ROOT / 'source_commit.txt').write_text(commit + '\n')

torch: 2.11.0+cu130
transformers: 4.57.6
datasets: 4.8.5
cuda: 13.0


41

## 4. Reusable stage runner with Drive-backed outputs

In [ ]:
import shlex

def run_stage(name, args, artifacts):
    """Restore a complete stage or run it and copy outputs to Drive.

    args are passed directly to Python (no shell interpretation).
    artifacts are paths relative to the repository.
    """
    stage_dir = DRIVE_ROOT / name
    stage_dir.mkdir(parents=True, exist_ok=True)
    local_paths = [REPO / p for p in artifacts]
    saved_paths = [stage_dir / p for p in artifacts]
    if all(p.is_file() for p in saved_paths):
        for dest, src in zip(local_paths, saved_paths):
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.is_file(): shutil.copy2(src, dest)
        print(f'RESTORED / SKIPPED completed stage: {name}')
        return
    if any(p.is_file() for p in saved_paths):
        raise RuntimeError(f'Partial Drive output for {name}; inspect {stage_dir} before retrying.')
    for p in local_paths:
        if p.exists():
            raise FileExistsError(f'Local output exists without recorded Drive stage: {p}')
    print('RUNNING:', name, shlex.join(args), flush=True)
    t0 = time.perf_counter()
    with (stage_dir / 'stdout.log').open('w') as log:
        proc = subprocess.Popen([sys.executable, '-u', '-m', *args],
                                cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    if rc != 0: raise RuntimeError(f'{name} failed with return code {rc}; see Drive stdout.log')
    for local, dest in zip(local_paths, saved_paths):
        if not local.is_file(): raise FileNotFoundError(f'Missing output: {local}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local, dest)
    (stage_dir / 'completed.json').write_text(json.dumps({
        'name': name, 'git_commit': commit, 'seconds': round(time.perf_counter() - t0, 2),
        'outputs': artifacts}, indent=2))
    print('SAVED completed stage to Drive:', stage_dir)

# A new Drive directory should be used for each new code revision or experimental run.

## 5. Train the V3 warm-up retriever

In [ ]:
WARM_DIR = 'experiments/v3/warmup/full'
WARM_BASE = 'dpr_warmup_n2000_b32_seed42_steps62'
WARM_PT = f'{WARM_DIR}/{WARM_BASE}.pt'
run_stage('01_warmup', ['src.warmup_v3', '--max-samples','2000',
  '--batch-size','32','--epochs','1','--seed','42','--output-dir',WARM_DIR],
  [f'{WARM_DIR}/{WARM_BASE}{suffix}' for suffix in ('.pt','.json','_steps.csv')])

RUNNING: 01_warmup src.warmup_v3 --max-samples 2000 --batch-size 32 --epochs 1 --seed 42 --output-dir experiments/v3/warmup/full


Device: cuda



Generating train split: 100%|██████████| 87599/87599 [00:00<00:00, 727224.00 examples/s]

Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 742758.90 examples/s]


Unique question/passage pairs: 2000
Steps per full epoch: 62
Optimizer steps: 62


I0000 00:00:1792847087.804982     965 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847087.876218     965 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

Step 50 | Scores (32, 32) | Loss 0.1810 | Elapsed 8.5s


Step 60 | Scores (32, 32) | Loss 0.2102 | Elapsed 9.9s


Step 62 | Scores (32, 32) | Loss 0.3218 | Elapsed 10.1s


Average loss: 0.7640060365440384
Training seconds: 10.148152371999998
CUDA peak allocated GB: 3.7079854011535645
Saved: experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62.pt
Metadata: experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62.json
Step log: experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62_steps.csv


SAVED completed stage to Drive: /content/dpr_v3_a100_seed42/01_warmup


## 6. Mine model-hard negatives from 5,000 passages

In [ ]:
MINE_DIR = 'experiments/v3/mining/full'
MINE_PT = f'{MINE_DIR}/mined_negatives.pt'
run_stage('02_mining', ['src.mine_negatives','--num-questions','2000',
  '--num-passages','5000','--num-hard','4','--batch-size','32',
  '--seed','42','--checkpoint',WARM_PT,'--output-dir',MINE_DIR],
  [MINE_PT, f'{MINE_DIR}/passage_index.pt', f'{MINE_DIR}/manifest.json'])

RUNNING: 02_mining src.mine_negatives --num-questions 2000 --num-passages 5000 --num-hard 4 --batch-size 32 --seed 42 --checkpoint experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62.pt --output-dir experiments/v3/mining/full


Device: cuda
Questions: 2000 Passages: 5000


I0000 00:00:1792847118.107836    1216 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847118.176413    1216 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792847120.578608    1216 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847120.579116    1216 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Encoded passages: 32/5000


...

Encoded passages: 4800/5000


Encoded questions: 32/2000
Encoded questions: 800/2000


Encoded questions: 1600/2000
Passage embeddings: (5000, 768)
Question embeddings: (2000, 768)


Hard-negative indices: (2000, 4)
Diagnostics: {
  "mean_positive_rank": 10.779500007629395,
  "median_positive_rank": 2.0,
  "positive_rank_1_count": 951,
  "negative_margin_count": 1049,
  "mean_positive_negative_margin": -0.30631765723228455,
  "recall_at_1": 0.4754999876022339,
  "recall_at_5": 0.7570000290870667,
  "recall_at_20": 0.9139999747276306,
  "mean_answer_rejections_per_query": 0.436,
  "total_answer_rejections": 872,
  "max_candidates_scanned_for_one_query": 129
}

Example question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Positive passage: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
Hard negative 1 (passage 3703, score 47.573): In the 1990s two icons by the Russian icon painter Sergei Fyodorov were hung in the abbey. On 6 September 1997 the funeral of Diana, Princess of Wales, was held

SAVED completed stage to Drive: /content/dpr_v3_a100_seed42/02_mining


### Mining sanity check
This checks positive exclusion and unique IDs. Exact answer overlap was tested in the local smoke run; it remains a heuristic and cannot rule out semantically relevant false negatives.

In [ ]:
mined = torch.load(REPO / MINE_PT, map_location='cpu', weights_only=True)
assert len(mined['questions']) == 2000
assert len(mined['passages']) == 5000
hard_ids = mined['hard_negative_indices']
assert tuple(hard_ids.shape) == (2000,4)
assert bool(((hard_ids >= 0) & (hard_ids < 5000)).all())
for i in range(2000):
    row = hard_ids[i].tolist()
    assert i not in row and len(set(row)) == 4
print('Mining ID sanity checks passed.')

Mining ID sanity checks passed.


## 7. Four second-stage strategies with identical question batches

In [ ]:
TRAIN_DIR = 'experiments/v3/training/full'
STRATEGIES = ['random','hard','mixed','inbatch']
CHECKPOINTS = {}
for strategy in STRATEGIES:
    k = 32 if strategy == 'inbatch' else 4
    base = f'dpr_{strategy}_k{k}_b33_seed42_steps60'
    args = ['src.train','--strategy',strategy,'--mined-path',MINE_PT,
      '--warmup-path',WARM_PT,'--output-dir',TRAIN_DIR,
      '--batch-size','33','--num-negatives',str(k),
      '--seed','42','--max-steps','60','--log-every','10']
    run_stage('03_train_' + strategy, args,
      [f'{TRAIN_DIR}/{base}{suffix}' for suffix in ('.pt','.json','_steps.csv')])
    CHECKPOINTS[strategy] = f'{TRAIN_DIR}/{base}.pt'
print('All strategy checkpoints:', CHECKPOINTS)

RUNNING: 03_train_random src.train --strategy random --mined-path experiments/v3/mining/full/mined_negatives.pt --warmup-path experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62.pt --output-dir experiments/v3/training/full --batch-size 33 --num-negatives 4 --seed 42 --max-steps 60 --log-every 10


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Strategy: random Seed: 42


I0000 00:00:1792847162.435158    1432 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847162.506698    1432 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792847165.059623    1432 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847165.060124    1432 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Step 1 | Scores (33, 5) | Loss 0.0003 | Elapsed 0.9s


...

Step 40 | Scores (33, 33) | Loss 0.0660 | Elapsed 6.0s


Step 50 | Scores (33, 33) | Loss 0.0124 | Elapsed 7.4s


Step 60 | Scores (33, 33) | Loss 0.3423 | Elapsed 8.8s


Total optimizer steps: 60
Query exposures: 1980
Passage encoder inputs: 1980
Average loss: 0.18558895785051088
Training seconds: 8.829601533000016
CUDA peak allocated GB: 3.781
Saved: experiments/v3/training/full/dpr_inbatch_k32_b33_seed42_steps60.pt
Metrics: experiments/v3/training/full/dpr_inbatch_k32_b33_seed42_steps60.json
Per-step log: experiments/v3/training/full/dpr_inbatch_k32_b33_seed42_steps60_steps.csv


SAVED completed stage to Drive: /content/dpr_v3_a100_seed42/03_train_inbatch
All strategy checkpoints: {'random': 'experiments/v3/training/full/dpr_random_k4_b33_seed42_steps60.pt', 'hard': 'experiments/v3/training/full/dpr_hard_k4_b33_seed42_steps60.pt', 'mixed': 'experiments/v3/training/full/dpr_mixed_k4_b33_seed42_steps60.pt', 'inbatch': 'experiments/v3/training/full/dpr_inbatch_k32_b33_seed42_steps60.pt'}


**Fairness note:** Same second-stage seed, 33 query examples per step, and 60 optimizer steps mean the models see matched query batches. Their passage encoder workloads and training costs differ. `mixed` also has more total negatives per query than the other strategies; it is not a quality-only comparison.

## 8. Evaluate held-out retrieval performance

In [ ]:
EVAL_DIR = 'experiments/v3/evaluation/full'
EVAL_OUT = f'{EVAL_DIR}/seed42_1000.csv'
ckpts = [f'warmup={WARM_PT}'] + [f'{s}={p}' for s,p in CHECKPOINTS.items()]
args = ['src.evaluate', *sum((['--checkpoint', spec] for spec in ckpts), []),
  '--max-samples','1000','--batch-size','16','--save-query-ranks',
  '--output',EVAL_OUT]
run_stage('04_evaluation', args, [EVAL_OUT,
    f'{EVAL_DIR}/seed42_1000_query_ranks.csv',
    f'{EVAL_DIR}/seed42_1000_metadata.json'])
import pandas as pd
display(pd.read_csv(REPO / EVAL_OUT))

RUNNING: 04_evaluation src.evaluate --checkpoint warmup=experiments/v3/warmup/full/dpr_warmup_n2000_b32_seed42_steps62.pt --checkpoint random=experiments/v3/training/full/dpr_random_k4_b33_seed42_steps60.pt --checkpoint hard=experiments/v3/training/full/dpr_hard_k4_b33_seed42_steps60.pt --checkpoint mixed=experiments/v3/training/full/dpr_mixed_k4_b33_seed42_steps60.pt --checkpoint inbatch=experiments/v3/training/full/dpr_inbatch_k32_b33_seed42_steps60.pt --max-samples 1000 --batch-size 16 --save-query-ranks --output experiments/v3/evaluation/full/seed42_1000.csv


Device: cuda


Validation pairs: 1000



===== WARMUP =====


I0000 00:00:1792847338.529044    2470 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792847338.597742    2470 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

Similarity matrix shape: (1000, 1000)
recall@1: 0.6090
recall@5: 0.8370
recall@20: 0.9500
mrr@10: 0.7069

===== MIXED =====


Similarity matrix shape: (1000, 1000)
recall@1: 0.6250
recall@5: 0.8470
recall@20: 0.9570
mrr@10: 0.7241

===== INBATCH =====


Similarity matrix shape: (1000, 1000)
recall@1: 0.5450
recall@5: 0.8360
recall@20: 0.9530
mrr@10: 0.6679

Results saved to experiments/v3/evaluation/full/seed42_1000.csv
Per-query ranks saved to experiments/v3/evaluation/full/seed42_1000_query_ranks.csv
Metadata saved to experiments/v3/evaluation/full/seed42_1000_metadata.json


SAVED completed stage to Drive: /content/dpr_v3_a100_seed42/04_evaluation


,strategy,checkpoint,recall@1,recall@5,recall@20,mrr@10
0,warmup,experiments/v3/warmup/full/dpr_warmup_n2000_b3...,0.521,0.813,0.947,0.643148
1,random,experiments/v3/training/full/dpr_random_k4_b33...,0.473,0.776,0.937,0.601267
2,hard,experiments/v3/training/full/dpr_hard_k4_b33_s...,0.609,0.837,0.950,0.706879
3,mixed,experiments/v3/training/full/dpr_mixed_k4_b33_...,0.625,0.847,0.957,0.724128
4,inbatch,experiments/v3/training/full/dpr_inbatch_k32_b...,0.545,0.836,0.953,0.667891


## 9. Share results

In [ ]:
print('All completed artifacts are in:', DRIVE_ROOT)
print('Share at minimum:')
print('  -', DRIVE_ROOT / '04_evaluation' / EVAL_OUT)
print('  -', DRIVE_ROOT / '04_evaluation' / f'{EVAL_DIR}/seed42_1000_query_ranks.csv')
print('  -', DRIVE_ROOT / '04_evaluation' / f'{EVAL_DIR}/seed42_1000_metadata.json')
print('  - all four 03_train_* metrics JSONs and stdout logs')
print('  - source_commit.txt and environment.txt')
print('For independent verification, keep warm-up, mining artifacts and model checkpoints on Drive.')

All completed artifacts are in: /content/dpr_v3_a100_seed42
Share at minimum:
  - /content/dpr_v3_a100_seed42/04_evaluation/experiments/v3/evaluation/full/seed42_1000.csv
  - /content/dpr_v3_a100_seed42/04_evaluation/experiments/v3/evaluation/full/seed42_1000_query_ranks.csv
  - /content/dpr_v3_a100_seed42/04_evaluation/experiments/v3/evaluation/full/seed42_1000_metadata.json
  - all four 03_train_* metrics JSONs and stdout logs
  - source_commit.txt and environment.txt
For independent verification, keep warm-up, mining artifacts and model checkpoints on Drive.
